# Week 12: Fine-Tuning Generation Models

Last week we fine-tuned BERT to put a review into one of two boxes. This week the model we fine-tune writes text, one token at a time, the way our Week 4 model did.

The running task: someone at AeroBeat (Sara's headphone shop from Weeks 8 to 10) types a question in plain English, our model writes a SQL query, and the shop's database answers it. We picked SQL because it separates two things that are easy to mix up:

- **Fine-tuning teaches a behavior**: a task, an output format, the conventions of one database.
- **Facts come from somewhere else**: retrieval (Week 9) or a database. The model never has to remember how many orders Detroit placed. It only has to write the query that counts them.

## A: Teaching a New Output Format

### Week 11 vs Week 12

| | Week 11: BERT | Week 12: Qwen |
|---|---|---|
| Output | one label | a sequence of tokens (a SQL query) |
| Head | a new linear head, 1,538 weights | the language-model head it already has (Week 4) |
| Loss | cross-entropy on one label | cross-entropy on every token of the answer (Week 4) |
| Weights trained | all 109.5M | about 1.1M (LoRA, Class 2) |
| Metric | accuracy | does the generated query return the right rows? |

Nothing about the model changes. A generation model already ends in a layer that scores every token in the vocabulary, so fine-tuning only changes which tokens it prefers after a given prompt.

### The AeroBeat Orders Database, and SQL in Five Minutes

One table, 120 orders placed between July 1 and October 28, 2026, loaded from the course repository (`data/aerobeat_orders.csv`). Order 1001 is Sara's Pro 2 from her Week 10 receipt.

We use four pieces of SQL this week, nothing else:

| Piece | Example | What it does |
|---|---|---|
| `SELECT ... FROM` | `SELECT customer FROM orders` | which column to return |
| `WHERE` | `WHERE city = 'Detroit' AND status = 'returned'` | which rows to keep; `LIKE '2026-10%'` matches any October date |
| `COUNT(*)` | `SELECT COUNT(*) FROM orders WHERE ...` | how many rows matched |
| `ORDER BY ... LIMIT` | `ORDER BY order_date DESC LIMIT 3` | newest first, keep 3 |

Python's built-in `sqlite3` keeps the table in memory, so there is nothing to install. `SCHEMA` is kept as a string because the same text goes into Qwen's prompt later. `run_sql()` returns the rows sorted, so two different queries can be compared by their results, and it returns an error message instead of crashing when a query is broken.

In [ ]:
import sqlite3
import pandas as pd

DATA = "https://raw.githubusercontent.com/razi-cmu/CPS535/main/data/"

SCHEMA = ("CREATE TABLE orders (order_id INTEGER, customer TEXT, product TEXT, "
          "city TEXT, order_date TEXT, status TEXT)")
db = sqlite3.connect(":memory:")                     # a database in RAM, nothing to install
db.execute(SCHEMA)
orders = pd.read_csv(DATA + "aerobeat_orders.csv")
orders.to_sql("orders", db, if_exists="append", index=False)

def run_sql(query):
    """Run a query and return its rows in a fixed order, or the error message."""
    try:
        return sorted(db.execute(query).fetchall(), key=str)
    except Exception as e:
        return f"ERROR: {e}"

print(orders.head(), "\n")
for q in ["SELECT COUNT(*) FROM orders WHERE city = 'Detroit'",
          "SELECT order_id, order_date FROM orders WHERE customer = 'Sara'",
          "SELECT order_id FROM orders WHERE order_date LIKE '2026-10%' ORDER BY order_date DESC LIMIT 3"]:
    print(q, "\n   ->", db.execute(q).fetchall())

### The Training and Test Pairs

A training pair is a prompt and the answer we want: a question in plain English and its correct SQL query. The pairs are ready-made CSV files in the course repository. How they were made:

- **From 10 question templates**, filled with real values from the table. Each template has three wordings for training and a fourth kept only for testing.
- **In people's words, not the table's.** The SQL always uses the stored value; the question often does not:

| The question says | The table stores |
|---|---|
| "Pro 2" | `AeroBeat Pro 2` |
| "sent back" | `returned` |
| "on the way" | `shipped` |
| "canceled" | `cancelled` |

These mappings are the conventions of this database that the model has to learn.

- **Every gold query was run** and kept only if it returns rows. If a gold answer were empty, a broken query that also returns nothing would be marked correct.
- **The test set cannot be memorized.** Test questions use value combinations never seen in training and the held-out fourth wording. The last line of the cell checks that no test query appears in training.

`load_pairs()` reads any CSV with a question column and an answer column. For your project, point it at your own file.

**The metric: execution accuracy.** Run the predicted query and the gold query, then compare the rows. Comparing the SQL text would be too strict:

| Gold | Prediction | Same text? | Same rows? |
|---|---|---|---|
| `WHERE order_date LIKE '2026-10%'` | `WHERE order_date BETWEEN '2026-10-01' AND '2026-10-31'` | no | yes |

In [ ]:
def load_pairs(source, question_col="question", sql_col="sql"):
    """Load prompt/response pairs from any CSV (swap in your own project's file here)."""
    df = pd.read_csv(source)
    return [{"question": q, "sql": s} for q, s in zip(df[question_col], df[sql_col])]

TRAIN = load_pairs(DATA + "aerobeat_sql_train.csv")
TEST  = load_pairs(DATA + "aerobeat_sql_test.csv")
print(f"{len(TRAIN)} training pairs, {len(TEST)} test pairs\n")

for p in TRAIN[:3] + TEST[:2]:
    print(f"Q: {p['question']}\n   SQL: {p['sql']}\n   rows: {run_sql(p['sql'])[:4]}")

assert not {p["sql"] for p in TEST} & {p["sql"] for p in TRAIN}   # no test query appears in training

381 training pairs and 46 test pairs. Notice that one SQL query appears several times in training with different wordings: that is the variety we want the model to learn from.

### Asking Qwen Zero-Shot

The Week 8 setup comes back unchanged: Qwen2.5-1.5B-Instruct in fp16, greedy decoding, and `generate()`. Three small helpers are new:

- `sql_prompt()` puts the question first, then the table, the instruction last (the order that worked in Week 9).
- `clean_sql()` strips Markdown fences and anything after the first `;`, so the model is not punished for formatting.
- `ask()` runs one question and prints every stage: the raw reply, the cleaned query, its rows, and, if we give it the gold query, the gold rows and whether they match.

Greedy decoding also sets `repetition_penalty` to 1.0. Qwen ships with 1.05, which quietly penalizes any token already used, and SQL repeats tokens such as `'` and `=` on purpose.

Before you run it, make a guess: the table's columns are right there in the prompt. Will a 1.5B instruction model get these three test questions right?

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

assert torch.cuda.is_available(), "Runtime > Change runtime type > T4 GPU"
DEVICE = "cuda"
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float16).to(DEVICE).eval()
gc = model.generation_config                       # greedy decoding, as in Weeks 7-8
gc.do_sample, gc.temperature, gc.top_p, gc.top_k, gc.repetition_penalty = False, None, None, None, 1.0

def chat_text(messages):
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def generate(messages, max_new_tokens=64):         # Week 8's generate()
    enc = tok(chat_text(messages), return_tensors="pt").to(DEVICE)
    out = model.generate(**enc, max_new_tokens=max_new_tokens)
    return tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)

def sql_prompt(question):                          # question first, then the table, instruction last (Week 9)
    return [{"role": "user", "content": f"Question: {question}\n\nTable:\n{SCHEMA}\n\n"
             "Write one SQLite query that answers the question. Reply with only the query."}]

def clean_sql(reply):                              # remove ``` fences and anything after the first ;
    return reply.replace("```sql", "").replace("```", "").strip().split(";")[0].strip()

def ask(question, gold=None):
    reply = generate(sql_prompt(question))
    query = clean_sql(reply)
    print(f"Q:      {question}")
    print(f"reply:  {reply!r}")                    # the raw reply, fences and newlines visible
    print(f"SQL:    {query}")
    print(f"rows:   {run_sql(query)}")
    if gold:
        print(f"gold:   {gold}")
        print(f"rows:   {run_sql(gold)}")
        print(f"same rows? {run_sql(query) == run_sql(gold)}")
    print()

for p in TEST[:3]:
    ask(p["question"], p["sql"])

Look at the `reply:` lines first. That is exactly what the model wrote, so you can see why `clean_sql()` exists.

Now judge each question yourself: is it right, and if not, which kind of miss is it?

1. **Part of the question ignored.** A condition is dropped, for example counting all returned orders when the question asked only about the Sport.
2. **Value conventions.** The person's words used as table values, such as `product = 'Pro 2'` or `status = 'on the way'`: valid SQL, zero rows, because the table stores `AeroBeat Pro 2` and `shipped`.
3. **The wrong column.** `SELECT *` when the question asked only for a name or an ID.
4. **Text around the query**, such as a sentence before it, which makes the query crash.

Also look for a query written differently from the gold one that still returns the same rows, for example `COUNT(order_id)` instead of `COUNT(*)`. It counts as correct.

**Execution accuracy** is the check you just did by hand, "same rows?", run on all 46 test questions and counted. Different SQL that returns the same rows counts as correct.

In [ ]:
def execution_accuracy(pairs):
    correct = 0
    for p in pairs:
        pred = clean_sql(generate(sql_prompt(p["question"])))
        correct += run_sql(pred) == run_sql(p["sql"])   # the same check as ask(), counted
    print(f"Execution accuracy: {correct}/{len(pairs)} = {correct / len(pairs):.1%}")
    return correct / len(pairs)

acc_before = execution_accuracy(TEST)

This number is the "before". We will compare against it in Class 2.

The model knows SQL. It does not know *this* database. That is the naive version. Prompt engineering could close part of the gap (Try-it 1); we will teach the conventions by fine-tuning instead.

### Where We Stand

We now know the problem: plain Qwen knows SQL but not our database. The fix is to fine-tune it on our pairs. But updating every weight the way Week 11 did needs, for Qwen's 1.54 billion weights with AdamW in fp32:

| What is stored | Bytes per weight | Total |
|---|---|---|
| weights | 4 | 6.2 GB |
| gradients | 4 | 6.2 GB |
| AdamW's two running averages | 8 | 12.3 GB |
| **total** | **16** | **24.6 GB** |

A Colab T4 has 15 GB. Week 8's memory table showed only the first row. Class 2 trains a tiny slice of the model instead.

## B: LoRA

### LoRA Intuition

Fine-tuning changes a weight matrix $W$. Full fine-tuning updates every number in it, and that is what did not fit in memory. **LoRA** (Low-Rank Adaptation) freezes $W$ and learns the change as two thin matrices, $A$ and $B$:

$$y = Wx + B(Ax)$$

Only $A$ and $B$ are trained. Two questions: where does training start, and how much is trained?

**Step 1: LoRA starts as the model we already have.** One $2 \times 2$ layer, rank 1, all values hand-picked:

$$W = \begin{bmatrix} 1 & 0 \\ 0 & 1 \end{bmatrix}, \quad x = \begin{bmatrix} 0.6 \\ 0.8 \end{bmatrix}, \quad A = \begin{bmatrix} 0.5 & 0.5 \end{bmatrix}, \quad B = \begin{bmatrix} 0 \\ 0 \end{bmatrix}$$

- $Ax = 0.5 \cdot 0.6 + 0.5 \cdot 0.8 = 0.3 + 0.4 = 0.7$
- $B(Ax) = [0 \cdot 0.7,\ 0 \cdot 0.7] = [0, 0]$
- $y = Wx + [0, 0] = [0.6, 0.8]$, exactly what the layer gave before LoRA.

$B$ starts at zero on purpose: adding LoRA changes nothing, so training starts from pretrained Qwen and not from noise. (A random $A$ lets $B$ start moving after the first step. If both started at zero, neither would ever move.)

**Step 2: how much is trained.** For a layer with $d_{in}$ inputs and $d_{out}$ outputs, full fine-tuning trains $d_{out} \times d_{in}$ numbers; LoRA with rank $r$ trains $r \times (d_{in} + d_{out})$. For Qwen's `q_proj` ($1536 \times 1536$) with $r = 8$:

- full: $1536 \times 1536 = 2{,}359{,}296$
- LoRA: $8 \times (1536 + 1536) = 8 \times 3072 = 24{,}576$, about 1%

We add LoRA to `q_proj` and `v_proj` in all 28 layers, about 1.1 million trained weights, or 0.07% of Qwen. With AdamW's 16 bytes each that is about 17 MB on top of the frozen 3.1 GB model. It fits on a T4.

In the toy we picked $A$ by hand; in Qwen both $A$ and $B$ are learned by backprop.

### LoRA in Code

`LoRALinear` wraps one existing layer: `base` is the frozen $W$, and `A` and `B` are the new trainable matrices. `forward()` is the board formula, $Wx + B(Ax)$. (The extra `scale` is a fixed multiplier, 2 here, so you can try another rank without retuning the learning rate.)

Then we freeze all of Qwen and wrap `q_proj` and `v_proj` in every layer. The last lines check Step 1 on the real model: the logits for one prompt before and after adding LoRA.

In [ ]:
import math
import torch.nn as nn

class LoRALinear(nn.Module):
    def __init__(self, base, r=8, alpha=16):
        super().__init__()
        self.base = base                                                      # frozen W
        self.A = nn.Parameter(torch.randn(r, base.in_features, device=DEVICE) / math.sqrt(base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r, device=DEVICE))  # B = 0
        self.scale = alpha / r
    def forward(self, x):
        return self.base(x) + (self.scale * (x.float() @ self.A.T) @ self.B.T).to(x.dtype)   # Wx + B(Ax)

x = tok(chat_text(sql_prompt(TEST[0]["question"])), return_tensors="pt")["input_ids"].to(DEVICE)
with torch.no_grad():
    before = model(x).logits

for p in model.parameters():
    p.requires_grad = False                                                   # freeze all of Qwen
for layer in model.model.layers:
    layer.self_attn.q_proj = LoRALinear(layer.self_attn.q_proj)
    layer.self_attn.v_proj = LoRALinear(layer.self_attn.v_proj)

with torch.no_grad():
    after = model(x).logits
print(f"trainable weights: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print("largest change in any logit (B = 0):", (after - before).abs().max().item())

The largest change is exactly 0.0, as the board predicted: with $B = 0$, adding LoRA changes nothing.

### Fine-Tuning Qwen to Write SQL

**What the model is graded on.** A training example is the zero-shot prompt followed by the gold SQL and `<|im_end|>`, the same chat format Qwen is used with. The loss is Week 4's cross-entropy with one change: **only the answer is graded.** Every prompt token gets the target `-100`, which `F.cross_entropy` skips. At use time the prompt is always given, so there is nothing to learn from it, and it is long and repeats the same table in every example.

The printout shows what is graded in one example: only the SQL and the final `<|im_end|>`, which teaches the model to stop after the query. Every prompt position has target `-100`.

**The loop** is Week 11's, simplified: one example per step (no padding needed), 2 passes over the 381 pairs, shuffled each time. (No `model.train()` / `model.eval()`: Qwen2.5 has no dropout, so they would change nothing. Keep them for models that do, like BERT.) Two differences from Week 11:

- **A larger learning rate** (1e-4 vs 2e-5). Only the small LoRA matrices move and Qwen's own weights are frozen, so there is no risk of wiping out what it knows.
- **`GradScaler`.** Qwen's weights are fp16, where very small gradients round to zero. The scaler multiplies the loss before `backward()` and divides the gradients back before the update.

To see the learning, not just a loss number, every 100 steps the cell asks the model one fixed test question and prints its current query, marked right or wrong. Then the same `execution_accuracy()` on the same 46 test questions, printed under the Class 1 number.

In [ ]:
import random
import torch.nn.functional as F

def build_example(question, sql):
    prompt = chat_text(sql_prompt(question))
    n_prompt = len(tok(prompt)["input_ids"])                                   # how many prompt tokens
    enc = tok(prompt + sql + "<|im_end|>", return_tensors="pt").to(DEVICE)     # prompt + answer, as in generate()
    targets = enc["input_ids"].clone()
    targets[0, :n_prompt] = -100                                               # grade only the answer
    return enc, targets

enc, targets = build_example(TRAIN[0]["question"], TRAIN[0]["sql"])
print(enc["input_ids"].shape[1], "tokens; graded part:", repr(tok.decode(enc["input_ids"][targets != -100])))

opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4)
scaler = torch.amp.GradScaler(DEVICE)          # fp16 model: keeps tiny gradients from rounding to 0
watch = TEST[2]                                # one test question to watch while training
print("watching:", watch["question"], "\ngold:    ", watch["sql"], "\n")

random.seed(0)
for epoch in range(2):
    random.shuffle(TRAIN)
    for step, p in enumerate(TRAIN):
        enc, targets = build_example(p["question"], p["sql"])
        logits = model(**enc).logits
        loss = F.cross_entropy(logits[0, :-1].float(), targets[0, 1:])   # Week 4 shift; -100 is skipped
        opt.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        if step % 100 == 0:
            pred = clean_sql(generate(sql_prompt(watch["question"])))
            mark = "right" if run_sql(pred) == run_sql(watch["sql"]) else "wrong"
            print(f"epoch {epoch} step {step:3d}  loss {loss.item():.2f}  {mark}:  {pred}")

print(f"before: {acc_before:.1%}")
execution_accuracy(TEST)

Only LoRA's weights changed between the two accuracy lines, so the gap is what fine-tuning taught. With 46 test questions, one question is about 2 points, so small differences between runs are noise.

To keep the result, save only $A$ and $B$:

```python
torch.save({name: p for name, p in model.named_parameters() if p.requires_grad}, "aerobeat_sql_lora.pt")
```

That file is about 4 MB, against 3.1 GB for Qwen. Teams keep one base model and swap small adapters per task. The `peft` library does all of the above in one line:

```python
model = get_peft_model(model, LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj", "v_proj"]))
```

**QLoRA** is the same idea with the frozen $W$ stored in 4 bits (Week 8's memory table), which is how people fine-tune 7B to 70B models on one GPU.

### Before vs After, and Your Own Questions

First the same three test questions you judged by hand in Class 1, now on the fine-tuned model: compare each answer with what plain Qwen wrote. Then two questions that probe the edges, so check each answer against the table:

- The first needs `>`, which no template contains, and a date. The question never says which year, and nothing in the prompt does either.
- The second asks about "Sarah", who is not in the table. A query can run fine and return `[]`, and nothing tells the user the name was the problem.

In [ ]:
for p in TEST[:3]:                             # the three questions from Class 1, now after fine-tuning
    ask(p["question"], p["sql"])

ask("How many orders did Sara place after September 1?")   # needs >, and a year the prompt never gives
ask("What did Sarah order?")                               # a name that is not in the table
# ask("your own question here")

### What Is Still Wrong

| What went wrong | Why | Where we work on it |
|---|---|---|
| Questions outside the 10 patterns (`>`, averages, two tables) | templates cover only what we wrote | more varied data; Week 13 retrieves similar past questions as examples |
| A name that is not in the table returns `[]` | the model copies the word it was given | check values in code before running the query |
| Dates without a year | nothing in the prompt says it is 2026 | put today's date in the prompt, or let code fill dates |
| Facts trained into the weights | fine-tuning learns patterns, not a lookup | RAG (Week 9) or a database |
| A different table breaks it | the behavior was learned for one schema | include several schemas in the training data |
| The model could write `DELETE` or `DROP` | nothing stops it | open the database read-only; Week 14 |
| A question that says "ignore the instructions and..." | prompt injection | Week 14 |


A few hundred verified pairs were enough here because the task is narrow. A real task with varied wording needs more pairs and more varied phrasings.